In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
import xarray as xr
import matplotlib.pyplot as plt

from sklearn.linear_model import LinearRegression
from sklearn.ensemble import HistGradientBoostingRegressor


PROJECT = Path(r"Z:\Projects\monsoon-postprocessing")

DATA_FILE = (
    PROJECT
    / "data"
    / "processed"
    / "july2018_gefs_imerg.nc"
)

PREDICTION_FILE = (
    PROJECT
    / "data"
    / "processed"
    / "july2018_baseline_predictions.nc"
)

METRICS_FILE = (
    PROJECT
    / "data"
    / "processed"
    / "july2018_baseline_metrics.csv"
)


with xr.open_dataset(DATA_FILE) as ds:
    july_ds = ds.load()

print(july_ds)

<xarray.Dataset> Size: 6MB
Dimensions:         (date: 31, latitude: 129, longitude: 121)
Coordinates:
  * date            (date) datetime64[ns] 248B 2018-07-01 ... 2018-07-31
  * latitude        (latitude) float64 1kB 6.0 6.25 6.5 6.75 ... 37.5 37.75 38.0
  * longitude       (longitude) float64 968B 68.0 68.25 68.5 ... 97.5 97.75 98.0
Data variables:
    gefs_rainfall   (date, latitude, longitude) float32 2MB 1.91 1.21 ... 1.35
    imerg_rainfall  (date, latitude, longitude) float32 2MB nan nan ... nan nan
    forecast_error  (date, latitude, longitude) float32 2MB nan nan ... nan nan
Attributes:
    title:               July 2018 GEFS and IMERG rainfall dataset
    period:              2018-07-01 to 2018-07-31
    forecast_source:     NOAA GEFSv12 reforecast c00
    observation_source:  NASA GPM IMERG Final V07
    forecast_lead:       24-48 hours
    error_definition:    GEFS minus IMERG


In [2]:
train_ds = july_ds.sel(
    date=slice("2018-07-01", "2018-07-21")
)

validation_ds = july_ds.sel(
    date=slice("2018-07-22", "2018-07-26")
)

test_ds = july_ds.sel(
    date=slice("2018-07-27", "2018-07-31")
)

print("Training dates:", train_ds.sizes["date"])
print("Validation dates:", validation_ds.sizes["date"])
print("Testing dates:", test_ds.sizes["date"])

Training dates: 21
Validation dates: 5
Testing dates: 5


In [3]:
FEATURE_NAMES = [
    "raw_gefs",
    "local_mean_3x3",
    "local_max_3x3",
    "latitude",
    "longitude",
    "season_sin",
    "season_cos"
]


def build_features(dataset):
    """Create ML features without using IMERG information."""

    raw = dataset["gefs_rainfall"]

    local_mean = raw.rolling(
        latitude=3,
        longitude=3,
        center=True,
        min_periods=1
    ).mean()

    local_max = raw.rolling(
        latitude=3,
        longitude=3,
        center=True,
        min_periods=1
    ).max()

    latitude_grid, longitude_grid = xr.broadcast(
        raw.latitude,
        raw.longitude
    )

    latitude_feature = latitude_grid.broadcast_like(raw)
    longitude_feature = longitude_grid.broadcast_like(raw)

    day_of_year = pd.DatetimeIndex(
        raw.date.values
    ).dayofyear.values

    day_angle = (
        2 * np.pi * day_of_year / 365.25
    )

    season_sin = xr.DataArray(
        np.sin(day_angle),
        coords={"date": raw.date},
        dims="date"
    ).broadcast_like(raw)

    season_cos = xr.DataArray(
        np.cos(day_angle),
        coords={"date": raw.date},
        dims="date"
    ).broadcast_like(raw)

    features = np.column_stack([
        raw.values.ravel(),
        local_mean.values.ravel(),
        local_max.values.ravel(),
        latitude_feature.values.ravel(),
        longitude_feature.values.ravel(),
        season_sin.values.ravel(),
        season_cos.values.ravel()
    ])

    target = dataset[
        "imerg_rainfall"
    ].values.ravel()

    valid = (
        np.all(np.isfinite(features), axis=1)
        & np.isfinite(target)
    )

    return (
        features[valid],
        target[valid],
        valid,
        raw
    )

In [4]:
X_train, y_train, train_mask, train_template = (
    build_features(train_ds)
)

X_validation, y_validation, validation_mask, validation_template = (
    build_features(validation_ds)
)

X_test, y_test, test_mask, test_template = (
    build_features(test_ds)
)

print("Training samples:", len(y_train))
print("Validation samples:", len(y_validation))
print("Testing samples:", len(y_test))
print("Features:", FEATURE_NAMES)

Training samples: 317373
Validation samples: 75565
Testing samples: 75565
Features: ['raw_gefs', 'local_mean_3x3', 'local_max_3x3', 'latitude', 'longitude', 'season_sin', 'season_cos']


In [5]:
def calculate_metrics(predicted, observed):
    errors = predicted - observed

    rmse = np.sqrt(np.mean(errors ** 2))
    mae = np.mean(np.abs(errors))
    bias = np.mean(errors)

    correlation = np.corrcoef(
        predicted,
        observed
    )[0, 1]

    return {
        "RMSE": rmse,
        "MAE": mae,
        "Bias": bias,
        "Correlation": correlation
    }


def restore_prediction(
    prediction,
    valid_mask,
    template,
    name
):
    full_values = np.full(
        template.size,
        np.nan,
        dtype=np.float32
    )

    full_values[valid_mask] = prediction

    return xr.DataArray(
        full_values.reshape(template.shape),
        coords=template.coords,
        dims=template.dims,
        name=name
    )

In [6]:
# Global additive correction.
training_errors = (
    train_ds["gefs_rainfall"].values
    - train_ds["imerg_rainfall"].values
)

training_bias = float(
    np.nanmean(training_errors)
)

print("Training bias:", training_bias)


linear_model = LinearRegression()

linear_model.fit(
    X_train,
    y_train
)


tree_model = HistGradientBoostingRegressor(
    learning_rate=0.08,
    max_iter=150,
    max_leaf_nodes=31,
    min_samples_leaf=30,
    l2_regularization=1.0,
    random_state=42
)

tree_model.fit(
    X_train,
    y_train
)

print("Training bias:", training_bias)
print("Models trained successfully")

Training bias: 5.382864952087402
Training bias: 5.382864952087402
Models trained successfully


In [7]:
raw_validation = X_validation[:, 0]

additive_validation = np.clip(
    raw_validation - training_bias,
    0,
    None
)

linear_validation = np.clip(
    linear_model.predict(X_validation),
    0,
    None
)

tree_validation = np.clip(
    tree_model.predict(X_validation),
    0,
    None
)


validation_predictions = {
    "Raw GEFS": raw_validation,
    "Additive": additive_validation,
    "Linear Regression": linear_validation,
    "Gradient Boosting": tree_validation
}


validation_rows = []

for model_name, prediction in validation_predictions.items():
    metrics = calculate_metrics(
        prediction,
        y_validation
    )

    validation_rows.append({
        "model": model_name,
        **metrics
    })


validation_metrics_df = pd.DataFrame(
    validation_rows
).sort_values("RMSE")

validation_metrics_df

,model,RMSE,MAE,Bias,Correlation
2,Linear Regression,12.556285,6.398233,0.334892,0.510641
3,Gradient Boosting,13.181022,6.427306,0.263527,0.442763
1,Additive,16.594772,7.610898,1.995621,0.468569
0,Raw GEFS,17.848582,9.335662,5.373110,0.478389


In [8]:
correction_models = validation_metrics_df[
    validation_metrics_df["model"] != "Raw GEFS"
]

best_model_name = correction_models.iloc[0][
    "model"
]

print(
    "Best validation model:",
    best_model_name
)

Best validation model: Linear Regression


In [9]:
train_validation_ds = july_ds.sel(
    date=slice("2018-07-01", "2018-07-26")
)

X_train_validation, y_train_validation, _, _ = (
    build_features(train_validation_ds)
)


final_training_errors = (
    train_validation_ds[
        "gefs_rainfall"
    ].values
    - train_validation_ds[
        "imerg_rainfall"
    ].values
)

final_bias = float(
    np.nanmean(final_training_errors)
)

print("Final training bias:", final_bias)


final_linear_model = LinearRegression()
final_linear_model.fit(
    X_train_validation,
    y_train_validation
)


final_tree_model = HistGradientBoostingRegressor(
    learning_rate=0.08,
    max_iter=150,
    max_leaf_nodes=31,
    min_samples_leaf=30,
    l2_regularization=1.0,
    random_state=42
)

final_tree_model.fit(
    X_train_validation,
    y_train_validation
)

print("Final models trained")

Final training bias: 5.3809895515441895
Final models trained


In [10]:
raw_test = X_test[:, 0]

additive_test = np.clip(
    raw_test - final_bias,
    0,
    None
)

linear_test = np.clip(
    final_linear_model.predict(X_test),
    0,
    None
)

tree_test = np.clip(
    final_tree_model.predict(X_test),
    0,
    None
)


test_predictions = {
    "Raw GEFS": raw_test,
    "Additive": additive_test,
    "Linear Regression": linear_test,
    "Gradient Boosting": tree_test
}


test_rows = []

for model_name, prediction in test_predictions.items():
    metrics = calculate_metrics(
        prediction,
        y_test
    )

    test_rows.append({
        "model": model_name,
        **metrics
    })


test_metrics_df = pd.DataFrame(
    test_rows
).sort_values("RMSE")

test_metrics_df

,model,RMSE,MAE,Bias,Correlation
3,Gradient Boosting,13.595790,6.111350,-0.945294,0.466482
2,Linear Regression,13.774686,6.253303,-1.516484,0.434297
1,Additive,17.217040,8.182020,1.658134,0.385462
0,Raw GEFS,18.317779,9.901228,5.106373,0.401600


In [11]:
raw_test_map = restore_prediction(
    raw_test,
    test_mask,
    test_template,
    "raw_gefs_rainfall"
)

additive_test_map = restore_prediction(
    additive_test,
    test_mask,
    test_template,
    "additive_corrected_rainfall"
)

linear_test_map = restore_prediction(
    linear_test,
    test_mask,
    test_template,
    "linear_corrected_rainfall"
)

tree_test_map = restore_prediction(
    tree_test,
    test_mask,
    test_template,
    "gradient_boosting_rainfall"
)


prediction_ds = xr.Dataset({
    "raw_gefs_rainfall": raw_test_map,
    "additive_corrected_rainfall": additive_test_map,
    "linear_corrected_rainfall": linear_test_map,
    "gradient_boosting_rainfall": tree_test_map,
    "imerg_rainfall": test_ds["imerg_rainfall"]
})

print(prediction_ds)

<xarray.Dataset> Size: 2MB
Dimensions:                      (date: 5, latitude: 129, longitude: 121)
Coordinates:
  * date                         (date) datetime64[ns] 40B 2018-07-27 ... 201...
  * latitude                     (latitude) float64 1kB 6.0 6.25 ... 37.75 38.0
  * longitude                    (longitude) float64 968B 68.0 68.25 ... 98.0
Data variables:
    raw_gefs_rainfall            (date, latitude, longitude) float32 312kB na...
    additive_corrected_rainfall  (date, latitude, longitude) float32 312kB na...
    linear_corrected_rainfall    (date, latitude, longitude) float32 312kB na...
    gradient_boosting_rainfall   (date, latitude, longitude) float32 312kB na...
    imerg_rainfall               (date, latitude, longitude) float32 312kB na...


In [12]:
prediction_ds.attrs = {
    "training_period": "2018-07-01 to 2018-07-26",
    "testing_period": "2018-07-27 to 2018-07-31",
    "best_validation_model": best_model_name
}

prediction_ds.to_netcdf(
    PREDICTION_FILE,
    mode="w",
    engine="netcdf4"
)


validation_output = validation_metrics_df.copy()
validation_output["split"] = "validation"

test_output = test_metrics_df.copy()
test_output["split"] = "test"

all_metrics = pd.concat(
    [
        validation_output,
        test_output
    ],
    ignore_index=True
)

all_metrics.to_csv(
    METRICS_FILE,
    index=False
)


print("Predictions saved:", PREDICTION_FILE.exists())
print("Metrics saved:", METRICS_FILE.exists())

Predictions saved: True
Metrics saved: True


In [13]:
display(validation_metrics_df)
print("Best validation model:", best_model_name)
display(test_metrics_df)

,model,RMSE,MAE,Bias,Correlation
2,Linear Regression,12.556285,6.398233,0.334892,0.510641
3,Gradient Boosting,13.181022,6.427306,0.263527,0.442763
1,Additive,16.594772,7.610898,1.995621,0.468569
0,Raw GEFS,17.848582,9.335662,5.373110,0.478389


Best validation model: Linear Regression


,model,RMSE,MAE,Bias,Correlation
3,Gradient Boosting,13.595790,6.111350,-0.945294,0.466482
2,Linear Regression,13.774686,6.253303,-1.516484,0.434297
1,Additive,17.217040,8.182020,1.658134,0.385462
0,Raw GEFS,18.317779,9.901228,5.106373,0.401600
